# Bacpipe Tutorial
This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.

---
## 1. Setup & Configuration
Import necessary modules and set the working directory to the repository root.

In [1]:
from IPython.display import display
import os 

# replace this with the path to the directory on your machine
os.chdir('../..')
import bacpipe

/home/siriussound/Code/bacpipe/.env_bacpipe/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/siriussound/Code/bacpipe/bacpipe/embedding_evaluation/visualization/dashboard.py:37: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension('plotly')


---
## 2. Using a Custom Model
Define your own model by subclassing `ModelBaseClass` and plug it directly into bacpipe's pipeline.

In [2]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000
    SEGMENT_LENGTH = 48000

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio * 10

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # return array needs to be 2D!
        mel_spec = mel_spec.reshape(
            [len(mel_spec), mel_spec.shape[-2] * mel_spec.shape[-1]]
            )
        return mel_spec

---
## 3. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [3]:
from bacpipe.model_pipelines.feature_extractors.birdnet import Model

class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input ** 2
        return self.embeds(input, training=False)

2026-04-09 19:32:20.740060: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 19:32:20.773919: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---
## 4. Probing Pipeline
Train a linear probe on top of frozen embeddings to evaluate how well the representations capture your specific ground truth annotations.

In [ ]:
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

embeds = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet2',
    use_folder_structure=True,
    CustomModel=MyBirdNETModel
).embeddings(return_type='array')

probe, label2idx, metrics = bacpipe.probing_pipeline('birdnet', gt, embeds)

display(type(probe))
display(label2idx)
display(metrics)


### Embeddings already exist. Using embeddings in bacpipe_results/test_data/embeddings/2026-04-09_19-17___birdnet-test_data ###
Found 7 samples in the train set with 3 unique labels.
Epoch 1/20
Epoch [1/20], Loss: 1.0800, Accuracy: 28.57%
Epoch 2/20
Epoch [2/20], Loss: 0.9768, Accuracy: 71.43%
Epoch 3/20
Epoch [3/20], Loss: 0.8868, Accuracy: 71.43%
Epoch 4/20
Epoch [4/20], Loss: 0.8091, Accuracy: 71.43%
Epoch 5/20
Epoch [5/20], Loss: 0.7418, Accuracy: 71.43%
Epoch 6/20
Epoch [6/20], Loss: 0.6831, Accuracy: 71.43%
Epoch 7/20
Epoch [7/20], Loss: 0.6309, Accuracy: 71.43%
Epoch 8/20
Epoch [8/20], Loss: 0.5837, Accuracy: 71.43%
Epoch 9/20
Epoch [9/20], Loss: 0.5403, Accuracy: 71.43%
Epoch 10/20
Epoch [10/20], Loss: 0.5000, Accuracy: 85.71%
Epoch 11/20
Epoch [11/20], Loss: 0.4623, Accuracy: 85.71%
Epoch 12/20
Epoch [12/20], Loss: 0.4270, Accuracy: 85.71%
Epoch 13/20
Epoch [13/20], Loss: 0.3940, Accuracy: 85.71%
Epoch 14/20
Epoch [14/20], Loss: 0.3632, Accuracy: 100.00%
Epoch 15/20
Epoch [15

bacpipe.embedding_evaluation.probing.train_probe.LinearProbe

{'Common Chaffinch': 0, 'Common Cuckoo': 1, 'Eurasian Blackbird': 2}

{'overall': {'macro_accuracy': 1.0, 'auc': 1.0, 'macro_f1': 1.0},
 'items_per_class': {'Common Chaffinch': 1,
  'Common Cuckoo': 1,
  'Eurasian Blackbird': 1},
 'per_class_accuracy': {'Common Chaffinch': 1.0,
  'Common Cuckoo': 1.0,
  'Eurasian Blackbird': 1.0},
 'config': {'main_results_dir': 'bacpipe_results',
  'embed_parent_dir': 'embeddings',
  'dim_reduc_parent_dir': 'dim_reduced_embeddings',
  'evaluations_dir': 'evaluations',
  'model_base_path': 'bacpipe/model_checkpoints',
  'global_batch_size': 8,
  'audio_suffixes': ['.wav', '.WAV', '.aif', '.mp3', '.MP3', '.flac', '.ogg'],
  'rm_embedding_on_keyboard_interrupt': True,
  'padding': 'wrap',
  'avoid_pipelined_gpu_inference': False,
  'nr_parallel_workers': False,
  'annotations_filename': 'annotations.csv',
  'only_embed_annotations': False,
  'default_label_keys': ['time_of_day',
   'day_of_year',
   'continuous_timestamp',
   'parent_directory',
   'audio_file_name'],
  'min_label_occurrences': 50,
  'bool_filter_labels': 

---
## 5. Probe Inference
Load a previously trained linear probe and run inference to generate new predictions from your embeddings.

In [5]:
probe, label2idx = bacpipe.prepare_probe_inference('birdnet')

predictions = bacpipe.run_probe_inference(
    'birdnet', 
    probe, 
    0.5, 
    embeds, 
    device='cuda', 
    return_binary_presence=False
)

---
## 6. Clustering Pipeline
Run bacpipe's built-in clustering evaluation pipeline to group embeddings and compare the clusters against your ground truth labels.

In [6]:
gt = bacpipe.ground_truth_by_model(
    'birdnet', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

embeds = bacpipe.Loader(
    'bacpipe/tests/test_data', 
    'birdnet',
    use_folder_structure=True
).embeddings(return_type='array')

clusterings, metrics = bacpipe.clustering_pipeline('birdnet', gt, embeds)


### Embeddings already exist. Using embeddings in bacpipe_results/test_data/embeddings/2026-04-09_19-17___birdnet-test_data ###
Multiple embeddings found for model birdnet in bacpipe_results/test_data/embeddings. Using the most recent path.
/home/siriussound/Code/bacpipe/.env_bacpipe/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
/home/siriussound/Code/bacpipe/.env_bacpipe/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:49: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_label = type_of_target(labels_true)
/home/siriussound/Code/bacpipe/.env_bacpipe/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:49: User

---
## 7. Custom Clustering & Evaluation
Inject your own custom clustering algorithms (like sklearn's KMeans) and evaluate their performance using metrics like the Silhouette score.

In [7]:
from sklearn.cluster import KMeans

# Make sure embeddings are generated
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

# Run custom clustering without ground truth
clustered_points = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}
)

# Run custom clustering with ground truth for alignment/evaluation
clustered_points_gt = bacpipe.run_clustering(
    embeds, 
    {'my_clustering': KMeans(n_clusters=3)}, 
    ground_truth=gt['label:species']
)

# Evaluate embedding separation using Silhouette score
metrics = bacpipe.eval_with_silhouette(
    embeds, 
    ground_truth=gt['label:species']
)




###### Generating embeddings using BIRDNET ######


### Embeddings already exist. Using embeddings in bacpipe_results/test_data/embeddings/2026-04-09_19-17___birdnet-test_data ###
